# Preprocessing 

### Import library

In [26]:
import pandas as pd 
import re
import string
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK data files
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/cyrof/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/cyrof/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/cyrof/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [27]:
# Load csv file into dataframe 
DATADIR = f"{os.path.abspath(os.path.join(os.getcwd(), os.pardir))}/dataset"
main_df = pd.read_csv(f"{DATADIR}/trump_insults_tweets.csv")

In [28]:
# Drop unnecessary columns
main_df = main_df.drop(columns=main_df.columns[0])

# Convert 'data' column to datetime
main_df['date'] = pd.to_datetime(main_df['date'])

# Combine all tweets into a single string 
all_tweets = ' '.join(main_df['tweet'])

main_df.head()

,date,target,insult,tweet
0,2014-10-09,thomas-frieden,fool,"Can you believe this fool, Dr. Thomas Frieden ..."
1,2014-10-09,thomas-frieden,DOPE,"Can you believe this fool, Dr. Thomas Frieden ..."
2,2015-06-16,politicians,all talk and no action,Big time in U.S. today - MAKE AMERICA GREAT AG...
3,2015-06-24,ben-cardin,It's politicians like Cardin that have destroy...,Politician @SenatorCardin didn't like that I s...
4,2015-06-24,neil-young,total hypocrite,"For the nonbeliever, here is a photo of @Neily..."


In [37]:
# function to clean text 
def clean_text(text):
    # convert text to lowercase
    text = text.lower()

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # tokenize the text
    tokens = word_tokenize(text)

    # remove stopwords 
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize the text
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # join tokens back into a single string 
    cleaned_text = ' '.join(tokens)

    return cleaned_text

In [39]:
# Apply the cleaning function to each tweet
main_df['cleaned_tweet'] = main_df['tweet'].apply(clean_text)

# display the cleaned data
main_df[['tweet', 'cleaned_tweet']].head()

,tweet,cleaned_tweet
0,"Can you believe this fool, Dr. Thomas Frieden ...",believe fool dr thomas frieden cdc stated anyo...
1,"Can you believe this fool, Dr. Thomas Frieden ...",believe fool dr thomas frieden cdc stated anyo...
2,Big time in U.S. today - MAKE AMERICA GREAT AG...,big time u today make america great politician...
3,Politician @SenatorCardin didn't like that I s...,politician senatorcardin didnt like said balti...
4,"For the nonbeliever, here is a photo of @Neily...",nonbeliever photo neilyoung office request—tot...


### Save Cleaned Data for Top2Vec
ONce the text is cleand, save it to a new CSV file for use with Top2Vec

In [40]:
# save the cleaned tweets to a new CSV file 
main_df[['cleaned_tweet']].to_csv(f"{DATADIR}/cleaned_tweets.csv", index=False)